In [ ]:
#Unit Test for Intersection
import precovis_v002 as precovis
import matplotlib.pyplot as plt
import time
import random

import unittest

In [ ]:
con = sql.connect("index.db")
frames = pd.read_sql("""SELECT * FROM frames""", con)

start = 58270
end = 59972
coords_df, transitions_orbit = precovis.objectToCoordsDF("2020 AV2", start, end, True)
obs_frames = frames[(frames["obscode"] == 'I41') & (frames["dataset_id"] == 'ztf')]

#The ra and dec for this test don't match the healpixel for this testing
#0. Matches HP and T, 1. Matches HP, 2. Matches HP and Before T, 3. Matches HP and After T, 4. Matches HP and T, 5. No Matches, 6-8. Matches Both (for same frame entry) (6. Lower Edge Match, 7. Middle-ish, 8. Higher Edge Match), 9. Matches T not HP
ra_test=[283.1881543053395, 283.26558726953, 283.34302598343635, 283.42053657025554, 283.4981826969866, 283.5760193336291, 283.42053657025554, 283.4981826969866, 283.5760193336291, -26.29363945177291]
dec_test=[-26.25141146375982, -26.260193650731303, -26.268804544536607, -26.277243122727718, -26.285516546448388, -26.29363945177291, -26.277243122727718, -26.285516546448388, -26.29363945177291, -26.29363945177291]
time_test=[59123.105158, 59971.599988248774, 59123.095058, 59123.115058, 59123.105178, 59971.79999412438, 59123.105058, 59123.105103, 59123.105232, 59123.105108]
healpixel_test=[7574, 7485, 7482, 7480, 7482, 8795, 7481, 7481, 7481, 9870]

unittest_coords_df = pd.DataFrame(
    {'ra': ra_test,
     'dec': dec_test,
     'time': time_test,
     'healpixel': healpixel_test
    })

unittest_frames = obs_frames.head(20)

In [ ]:

#Unit Tests
class TestIntersectionsMethods(unittest.TestCase):
    
    def test1(self):
        print("Test 1: Empty frames returns empty list and dataframe")
        pixels, intersections = precovis.plotIntercectingHealpixels("2020 AV2", start, end, dur, unittest_frames.iloc[1:1], unittest_coords_df.iloc[1:1], 'I41', 'ztf', version)
        #print(intersections)
        self.assertEqual(pixels, [], "Test 1a failed: The list isn't empty.")
        self.assertEqual(len(intersections), 0, "Test 1a failed: The dataframe isn't empty.")
        print()

    def test2(self):
        print("Test 2: A single intersection returns correct pixel and dataframe (single row frame and coords)")
        pixels, intersections = precovis.plotIntercectingHealpixels("2020 AV2", start, end, dur, unittest_frames.iloc[1:2], unittest_coords_df.iloc[4:5], 'I41', 'ztf', version)
        #print(intersections)
        self.assertEqual(pixels, [7482], "Test 2a failed.")
        self.assertEqual(len(intersections), 1, "Test 2b failed.")
        print()

    def test3(self):
        print("Test 3: Matching healpixels but not matching dates return empty list and dataframe")
        pixels, intersections = precovis.plotIntercectingHealpixels("2020 AV2", start, end, dur, unittest_frames, unittest_coords_df.iloc[1:3], 'I41', 'ztf', version)
        #print(intersections)
        self.assertEqual(pixels, [], "Test 3a failed: The list isn't empty.")
        self.assertEqual(len(intersections), 0, "Test 3b failed: The dataframe isn't empty.")
        print()

    def test4(self):
        print("Test 4: Matching dates but not matching healpixels return empty list and dataframe")
        pixels, intersections = precovis.plotIntercectingHealpixels("2020 AV2", start, end, dur, unittest_frames, unittest_coords_df.iloc[9], 'I41', 'ztf', version)
        #print(intersections)
        self.assertEqual(pixels, [], "Test 4a failed: The list isn't empty.")
        self.assertEqual(len(intersections), 0, "Test 4b failed: The dataframe isn't empty.")
        print()

    def test5(self):
        print("Test 5: Multiple intersections from a single frame")
        pixels, intersections = precovis.plotIntercectingHealpixels("2020 AV2", start, end, dur, unittest_frames, unittest_coords_df.iloc[6:9], 'I41', 'ztf', version)
        #print(intersections)
        self.assertEqual(pixels, [7481], "Test 5a failed.")
        self.assertEqual(len(intersections), 3, "Test 5b failed.")
        print()

    def test6(self):
        print("Test 6: Everything together")
        pixels, intersections = precovis.plotIntercectingHealpixels("2020 AV2", start, end, dur, unittest_frames, unittest_coords_df, 'I41', 'ztf', version)
        #print(intersections)
        self.assertEqual(pixels, [7481,7482,7574], "Test 6a failed")
        self.assertEqual(len(intersections), 5, "Test 6b failed.")
        print()

In [ ]:
#Time Test
def testIntersectionTime(version):
    n_rows = list(range(0,1000,100))
    
    times = []
    for n in n_rows:
        start_time = time.time()
        pixels, intersections = precovis.plotIntersectingHealpixels("2020 AV2", start, end, dur, frames.head(n), coords_df.head(n), 'I41', 'ztf', version)
        end_time = time.time()
        total_time = end_time - start_time
        times.append(total_time)
        
        color = random.randrange(0, 2**24)
        hex_color = hex(color)
        std_color = "#" + hex_color[2:]
        
        
    label = "Version "+str(version)
    plt.scatter(n_rows, times, color=std_color, s=5, label=label)
    plt.legend(loc="upper left")
    plt.title("Intersection Time Table")
    plt.xlabel("Number of Input Rows")
    plt.ylabel("Time (sec)") 

plt.show()

In [ ]:
#Runnning Tests
version = 1
unittest.main(argv=[''], verbosity=2, exit=False)
for v in range(1,4):
    testIntersectionTime(v)